In [0]:
spark.sql("USE workspace.meteo")
print("Lignes dans bronze :", spark.table("bronze_weather").count())
display(spark.sql("SELECT * FROM bronze_weather LIMIT 5"))
# On récupère bien les datas de la table bronze_weather
# ville / heure / temperature / precipitation

### filtre qualité

In [0]:
spark.sql("""
SELECT *
FROM bronze_weather
WHERE temperature IS NOT NULL
  AND temperature BETWEEN -60 AND 60
  AND precipitation >= 0
""").createOrReplaceTempView("etape1_valides")

print("Après filtre qualité :", spark.table("etape1_valides").count())
# Suppression de valeurs aberrantes dans le cas d'un problème de mesure
# Ici pas de valeur aberrante 

### suppression des doublons

In [0]:
spark.sql(
    """
        SELECT *, ROW_NUMBER() OVER (PARTITION BY ville, heure ORDER BY heure) AS rn FROM etape1_valides
    """).createOrReplaceTempView("etape2_numerotees")

display(spark.sql("SELECT * FROM etape2_numerotees ORDER BY ville, heure LIMIT 10"))
# Affichage de la table temporaire ville / heure / température / precipitation / rn 
# rn = rang dans lequel la ville et l'heure sont les mêmes (Doublon si rn > 1)
# On garde uniquement les lignes où rn = 1 (pas de doublon)

In [0]:
spark.sql(
    """
        SELECT * FROM etape2_numerotees WHERE rn = 1
    """).createOrReplaceTempView("etape3_dedoublonnees")

print("Après dédoublonnage :", spark.table("etape3_dedoublonnees").count())
# Creation de la table etape3_dedoublonnees : ville / heure / température / precipitation / rn
# On ne garde que les lignes si rn = 1 (pas de doublon)

In [0]:
spark.sql(
    """
        SELECT * FROM etape2_numerotees WHERE rn = 1
    """).createOrReplaceTempView("etape3_dedoublonnees")

print("Après dédoublonnage :", spark.table("etape3_dedoublonnees").count())
#Dedoulonnage = suppression des lignes où la ville et l'heure sont les mêmes
# On garde une ligne par ville et par heure
# Ici pas de doublon de ville/heure 

In [0]:
spark.sql(
    """
        SELECT *
        FROM etape3_dedoublonnees
        WHERE CAST(heure AS TIMESTAMP) <= current_timestamp()
    """).createOrReplaceTempView("etape4_passe")

print("Après suppression du futur :", spark.table("etape4_passe").count())
# CAST(heure AS TIMESTAMP) <= current_timestamp() => heure <= maintenant
# CAST = conversion en timestamp
# On garde les lignes où l'heure est inférieure à l'heure actuelle

In [0]:
spark.sql(
    """
        CREATE OR REPLACE TABLE silver_weather AS
        SELECT
            ville,
            CAST(heure AS TIMESTAMP) AS heure,
            CAST(temperature AS DOUBLE) AS temperature,
            CAST(precipitation AS DOUBLE) AS precipitation
        FROM etape4_passe
    """)

spark.table("silver_weather").printSchema()
# On crée la table silver_weather
# On garde les colonnes ville, heure, température, précipitation
# On convertit les colonnes en double

In [0]:
display(spark.sql(
    """
        SELECT ville, COUNT(*) AS nb, MIN(heure) AS debut, MAX(heure) AS fin
        FROM silver_weather
        GROUP BY ville
    """))

# On affiche la table silver_weather
# On compte le nombre de lignes par ville
# On affiche la date de début et de fin de la mesure